In [5]:
import os
import json
import time
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"

PROJECT_ROOT = Path.cwd().parent          # notebook lives in notebooks/, so go up one level
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513
]

In [7]:
def fetch_movie(movie_id, session, retries=3):
    """Fetch one movie (with cast & crew). Returns a dict, or None if it fails."""
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "append_to_response": "credits"}

    for attempt in range(1, retries + 1):
        try:
            response = session.get(url, params=params, timeout=10)
        except requests.RequestException as e:
            print(f"ID {movie_id}: network error ({e}), attempt {attempt}/{retries}")
            time.sleep(2)
            continue

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:  # too many requests: wait and retry
            wait = int(response.headers.get("Retry-After", 2))
            time.sleep(wait)
            continue

        # Any other error (e.g. 404 for an ID that doesn't exist): don't retry
        message = response.json().get("status_message", "unknown error")
        print(f"ID {movie_id}: skipped ({response.status_code} - {message})")
        return None

    print(f"ID {movie_id}: failed after {retries} attempts")
    return None


def fetch_movies(ids):
    """Fetch several movies and return a list of the ones that succeeded."""
    movies = []
    with requests.Session() as session:
        for movie_id in ids:
            data = fetch_movie(movie_id, session)
            if data is not None:
                movies.append(data)
    print(f"\nFetched {len(movies)} of {len(ids)} movies.")
    return movies

In [8]:
raw_movies = fetch_movies(movie_ids)

with open(RAW_DIR / "movies_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_movies, f, ensure_ascii=False, indent=2)

ID 0: skipped (404 - The resource you requested could not be found.)

Fetched 18 of 19 movies.


In [6]:
if response.status_code == 200:
    movie = response.json()
else:
    print("Request failed:", response.status_code, response.json().get("status_message"))

In [10]:
df_raw = pd.DataFrame(raw_movies)

print(df_raw.shape)
df_raw.head()

(18, 28)


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,181,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.246,29031,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,162,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.611,34862,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
2,False,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,...,136,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.240,21026,"{'cast': [{'adult': False, 'gender': 1, 'id': ..."
3,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",300000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",https://www.marvel.com/movies/avengers-infinit...,299536,tt4154756,[US],en,...,149,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,An entire universe. Once and for all.,Avengers: Infinity War,False,8.245,33173,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
4,False,/sCzcYW9h55WcesOqA12cgEr9Exw.jpg,None,200000000,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",https://www.paramountmovies.com/movies/titanic,597,tt0120338,[US],en,...,194,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Nothing on earth could come between them.,Titanic,False,7.901,27914,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."


In [13]:
df_raw.loc[0, "credits"].keys()

dict_keys(['cast', 'crew'])